<div align="center">
<img src="https://raw.githubusercontent.com/MohammedAly22/VoiceTuT-TTS/main/assets/VoiceTut-TTS-Banner.png" width="100%"/>

[![🤗 Model](https://img.shields.io/badge/🤗%20Model-yellow)](https://huggingface.co/mohammedaly22/VoiceTut-TTS) 
[![GitHub](https://img.shields.io/badge/GitHub-181717?logo=github)](https://github.com/MohammedAly22/VoiceTuT-TTS)
</div>

# 📊 VoiceTut-TTS — Evaluation

Measures **performance** (RTF, time-to-first-audio, peak VRAM) and **quality**
(WER via ASR round-trip, speaker similarity for cloning, UTMOS naturalness) on a small
Egyptian-Arabic + code-switching eval set, then prints a summary table you can paste into
the README / model card.

> Runtime → GPU. First run downloads the model + eval models (a few minutes).

In [ ]:
# install: OmniVoice backbone, VoiceTut, and eval deps
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q voicetut-tts jiwer librosa
# Whisper for the WER round-trip (ASR of the synthesized audio)
!pip install -q transformers torchaudio

In [ ]:
import os, time, json, numpy as np, torch
from voicetut_tts import VoiceTutTTS, GenerationParams

CKPT = os.environ.get("VOICETUT_CKPT", "mohammedaly22/VoiceTut-TTS")
tts = VoiceTutTTS.from_pretrained(CKPT)
SR = tts.sampling_rate
device = "cuda" if torch.cuda.is_available() else "cpu"
print("loaded", CKPT, "| device:", device, "| sr:", SR)

## 1. Eval set
Pure Egyptian, code-switching, and English — short/medium/long. Edit freely.

In [ ]:
EVAL = [
    # (text, speaker, lang)
    ("ازيك عامل ايه النهاردة؟ يا رب تكون كويس.", "Mohamed", "arz"),
    ("النهارده الجو حلو اوي، يلا نطلع نتمشى على الكورنيش شوية.", "Yasmin", "arz"),
    ("بصراحة اليوم كان طويل ومليان مشاوير، صحيت بدري ورحت الشغل وبعدها عديت على السوبر ماركت.", "Sayed", "arz"),
    ("عندي meeting بكرة الصبح ومعايا ال presentation بتاعتي.", "Asmaa", "arz"),
    ("النهارده هنعمل review لل code وبعدها هنبدأ ال deployment على ال server.", "Omar", "arz"),
    ("Hello, thank you for calling our support line, how can I help you today?", "Abdelrahman", "en"),
]
print(len(EVAL), "eval items")

## 2. Performance — RTF, TTFA, peak VRAM
RTF = synthesis_time / audio_duration (lower is better). TTFA from the streaming API.

In [ ]:
def reset_vram():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
def peak_vram_gb():
    return torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else float('nan')

params = GenerationParams(num_step=32)
reset_vram()
rtfs, wavs = [], []
for text, spk, lang in EVAL:
    t0 = time.time()
    wav = tts.synthesize(text, speaker=spk, language=lang, params=params)
    dt = time.time() - t0
    dur = len(wav) / SR
    rtfs.append(dt / dur)
    wavs.append((wav, text, spk, lang))
    print(f"  RTF {dt/dur:.3f}  ({dur:.1f}s audio in {dt:.1f}s)  | {spk}")

# time-to-first-audio via the streaming API (first sentence)
long_text = "اهلا بيكم. النهارده هنتكلم عن موضوع مهم جدا. الموضوع ده بيخص الانتاجية وازاي تنظم وقتك. يلا نبدأ."
t0 = time.time(); ttfa = None
for i, (sr, ch) in enumerate(tts.stream(long_text, speaker="Sayed", params=params)):
    if i == 0: ttfa = time.time() - t0; break

PERF = {
    "rtf_mean": float(np.mean(rtfs)),
    "rtf_min": float(np.min(rtfs)),
    "ttfa_s": float(ttfa),
    "peak_vram_gb": float(peak_vram_gb()),
    "sampling_rate": SR,
}
print("\nPERF:", json.dumps(PERF, indent=2))

## 3. Intelligibility — WER (ASR round-trip)
Transcribe each synthesized clip with Whisper and compare to the input text. Lower WER = clearer
pronunciation. (Arabic WER is sensitive to orthography; treat as a relative signal.)

In [ ]:
import re, jiwer
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               device=0 if torch.cuda.is_available() else -1)

def clean_ar(s):
    s = re.sub(r"[ً-ْ]", "", s)                       # strip diacritics
    s = re.sub(r"[^\w\s\u0600-\u06FF]", " ", s)        # punctuation
    return re.sub(r"\s+", " ", s).strip().lower()

wer_ar, wer_en = [], []
for wav, text, spk, lang in wavs:
    hyp = asr({"array": np.asarray(wav, dtype=np.float32), "sampling_rate": SR})["text"]
    w = jiwer.wer(clean_ar(text), clean_ar(hyp))
    (wer_en if lang == "en" else wer_ar).append(w)
    print(f"  WER {w:.2f} [{lang}] {spk}: {hyp[:60]}")

WER = {"wer_arz": float(np.mean(wer_ar)) if wer_ar else None,
       "wer_en": float(np.mean(wer_en)) if wer_en else None}
print("\nWER:", json.dumps(WER, indent=2))

## 4. Speaker similarity (zero-shot cloning)
Clone a reference, then measure cosine similarity between the cloned output and the reference
speaker embedding (ECAPA via SpeechBrain). Higher = better voice match.

In [ ]:
sim_score = None
try:
    !pip install -q speechbrain
    import torchaudio, torch.nn.functional as F
    from speechbrain.inference.speaker import EncoderClassifier
    enc = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                         run_opts={"device": device})
    def emb(wav, sr):
        x = torch.tensor(np.asarray(wav, dtype=np.float32))[None]
        if sr != 16000:
            x = torchaudio.functional.resample(x, sr, 16000)
        return enc.encode_batch(x).squeeze()

    # use a built-in speaker's reference as the clone target
    spk = tts.list_speakers()[0]
    ref_wav, ref_sr = torchaudio.load(spk.audio_path)
    cloned = tts.synthesize("دي جملة اختبار لاستنساخ الصوت بتاع المتحدث المرجعي.",
                            ref_audio=spk.audio_path, ref_text=spk.reference_text, params=params)
    e_ref = emb(ref_wav.mean(0).numpy(), ref_sr)
    e_gen = emb(cloned, SR)
    sim_score = float(F.cosine_similarity(e_ref[None], e_gen[None]).item())
    print("speaker similarity (cosine):", round(sim_score, 3))
except Exception as e:
    print("similarity skipped:", e)

SIM = {"speaker_similarity": sim_score}

## 5. Naturalness — UTMOS
Predicted MOS (1–5) via the UTMOS model. Higher = more natural.

In [ ]:
mos = None
try:
    predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
    scores = []
    for wav, *_ in wavs:
        x = torch.tensor(np.asarray(wav, dtype=np.float32))[None]
        scores.append(float(predictor(x, SR).item()))
    mos = float(np.mean(scores))
    print("UTMOS (mean):", round(mos, 2))
except Exception as e:
    print("UTMOS skipped:", e)

MOS = {"utmos": mos}

## 6. Summary

In [ ]:
R = {**PERF, **WER, **SIM, **MOS}
def fmt(v, suf=""):
    return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.2f}{suf}"

print("| Metric | Value |")
print("|---|---|")
print(f"| Real-time factor (RTF, mean) | {fmt(R['rtf_mean'])}× |")
print(f"| RTF (best) | {fmt(R['rtf_min'])}× |")
print(f"| Time-to-first-audio (streaming) | {fmt(R['ttfa_s'],'s')} |")
print(f"| Peak VRAM (fp16) | {fmt(R['peak_vram_gb'],' GB')} |")
print(f"| WER — Egyptian Arabic | {fmt(R['wer_arz'])} |")
print(f"| WER — English | {fmt(R['wer_en'])} |")
print(f"| Speaker similarity (cloning) | {fmt(R['speaker_similarity'])} |")
print(f"| Naturalness (UTMOS, 1–5) | {fmt(R['utmos'])} |")
print(f"| Sampling rate | {SR} Hz |")

with open("voicetut_eval_results.json", "w") as f:
    json.dump(R, f, indent=2, ensure_ascii=False)
print("\nsaved -> voicetut_eval_results.json")